In [36]:
import polars as pl
import matplotlib.pyplot as plt


In [37]:
filepath = "../telemetria.csv"
data_raw = pl.read_csv(filepath, try_parse_dates=True)

In [38]:
print(f"Dimensions: {data_raw.shape}")
data_raw.head()

Dimensions: (253750, 15)


tracking_id,vehicle_id,company_id,route_id,driver_id,event_time,received_at,latitude,longitude,speed,fuel_level,battery_level,odometer_km,passenger_count,event_type
str,str,str,str,str,datetime[μs],datetime[μs],f64,f64,f64,f64,f64,f64,f64,str
"""61781b27-58b5-49a1-a4b9-a5365b…","""VEH_0003""","""EMP_003""","""RUTA_007""","""EMP_003_DRV_048""",2024-03-04 13:08:26,2024-03-04 13:08:32,2.620703,-74.338697,19.98,72.28,81.14,78234.21,25.0,"""P20"""
"""427e574c-1d4b-440f-bb0b-d06d84…","""VEH_0028""","""EMP_003""","""RUTA_018""","""EMP_003_DRV_037""",2024-03-05 18:07:20,2024-03-05 18:07:24,9.252828,-75.637242,28.05,79.58,82.84,44825.01,41.0,"""P20"""
"""dcbd2c8a-3b4d-429a-8af1-352f96…","""VEH_0156""","""EMP_001""","""RUTA_007""","""EMP_001_DRV_067""",2024-03-04 18:05:01,2024-03-04 18:05:05,2.888177,-73.223336,30.59,80.87,75.87,19427.25,null,"""TR10"""
"""e883266c-e209-4280-96d1-bb4079…","""VEH_0035""","""EMP_005""","""RUTA_010""","""EMP_005_DRV_079""",2024-03-05 15:36:09,2024-03-05 15:36:15,9.828977,-75.51476,65.75,66.67,92.99,35270.85,null,"""T30"""
"""80df3b8f-896f-4def-a64d-c476d4…","""VEH_0237""","""EMP_002""","""RUTA_013""","""EMP_002_DRV_052""",2024-03-05 01:55:45,2024-03-05 01:55:51,4.358705,-74.397611,57.48,37.75,84.32,17255.92,null,"""TR10"""


In [39]:
null_counts = data_raw.null_count()
null_counts

tracking_id,vehicle_id,company_id,route_id,driver_id,event_time,received_at,latitude,longitude,speed,fuel_level,battery_level,odometer_km,passenger_count,event_type
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,1250,0,0,0,1250,0,1250,1250,0,0,0,0,202759,0


In [40]:
data_raw.select(["speed", "fuel_level", "battery_level", "odometer_km"]).describe()

statistic,speed,fuel_level,battery_level,odometer_km
str,f64,f64,f64,f64
"""count""",253750.0,253750.0,253750.0,253750.0
"""null_count""",0.0,0.0,0.0,0.0
"""mean""",40.115612,65.495955,84.979306,43304.807609
"""std""",27.88731,16.510312,8.671745,21675.218098
"""min""",0.11,18.95,70.0,0.0
"""25%""",26.49,54.57,77.44,25901.73
"""50%""",37.65,66.78,84.98,42345.52
"""75%""",50.23,77.9,92.5,62331.3
"""max""",450.0,99.87,100.0,80339.35


In [41]:
duplicate_rows = data_raw.filter(
    pl.struct(["tracking_id", "vehicle_id", "event_time", "event_type"]).is_duplicated()
)

print(f"Dimensions: {duplicate_rows.shape}")
duplicate_rows.head()

Dimensions: (7368, 15)


tracking_id,vehicle_id,company_id,route_id,driver_id,event_time,received_at,latitude,longitude,speed,fuel_level,battery_level,odometer_km,passenger_count,event_type
str,str,str,str,str,datetime[μs],datetime[μs],f64,f64,f64,f64,f64,f64,f64,str
"""09d9a76f-d854-4555-851d-83a4de…","""VEH_0117""","""EMP_002""","""RUTA_019""","""EMP_002_DRV_025""",2024-03-04 14:24:43,2024-03-04 14:25:42,8.56277,-73.390917,32.28,78.3,92.99,44265.77,null,"""T30"""
"""bf60a268-d027-4919-aa8a-4bf0da…","""VEH_0180""","""EMP_005""","""RUTA_040""","""EMP_005_DRV_018""",2024-03-06 01:02:47,2024-03-06 01:32:47,6.800084,-75.239284,23.07,53.56,88.08,8934.83,null,"""T30"""
"""e2274cb0-1e32-440c-b325-752910…","""VEH_0211""","""EMP_001""","""RUTA_032""","""EMP_001_DRV_026""",2024-03-04 21:57:34,2024-03-04 21:59:25,7.711067,-75.424013,12.17,80.32,95.26,43307.79,13.0,"""P20"""
"""d6806ec3-331e-405b-a0a8-1042f5…","""VEH_0217""","""EMP_002""","""RUTA_006""","""EMP_002_DRV_017""",2024-03-04 17:07:17,2024-03-04 16:31:17,4.36243,-76.000481,200.0,69.48,86.72,74085.53,null,"""T30"""
"""d845eb42-3a61-41e1-a8f1-0210c0…","""VEH_0159""","""EMP_004""","""RUTA_025""","""EMP_004_DRV_036""",2024-03-06 00:25:44,2024-03-06 00:28:26,10.327604,-73.867406,26.1,43.01,70.39,62770.18,null,"""TR10"""


In [ ]:
outside_time = data_raw.filter(pl.col("event_time") > pl.col("received_at"))
print(f"Dimensions: {outside_time.shape}")
outside_time.head()

Dimensions: (4852, 15)


tracking_id,vehicle_id,company_id,route_id,driver_id,event_time,received_at,latitude,longitude,speed,fuel_level,battery_level,odometer_km,passenger_count,event_type
str,str,str,str,str,datetime[μs],datetime[μs],f64,f64,f64,f64,f64,f64,f64,str
"""fdd153bd-55a1-417b-89f6-e76f1b…","""VEH_0087""","""EMP_002""","""RUTA_035""","""EMP_002_DRV_003""",2024-03-05 12:20:30,2024-03-05 11:39:30,3.094907,-73.611575,29.92,62.79,88.0,71538.95,null,"""TR10"""
"""d6806ec3-331e-405b-a0a8-1042f5…","""VEH_0217""","""EMP_002""","""RUTA_006""","""EMP_002_DRV_017""",2024-03-04 17:07:17,2024-03-04 16:31:17,4.36243,-76.000481,200.0,69.48,86.72,74085.53,null,"""T30"""
"""a8a1a004-7f79-417d-a8e5-c4fbca…","""VEH_0094""","""EMP_004""","""RUTA_019""","""EMP_004_DRV_038""",2024-03-06 00:50:27,2024-03-05 23:54:27,10.303306,-73.072462,45.55,65.06,97.9,42145.56,6.0,"""P20"""
"""e0819787-b236-4146-8538-f5f75a…","""VEH_0033""","""EMP_003""","""RUTA_037""","""EMP_003_DRV_077""",2024-03-05 11:18:24,2024-03-05 10:37:24,7.849807,-73.527167,38.73,67.98,84.69,12401.17,null,"""T30"""
"""74194ba7-c786-4559-a2c1-a4d10c…","""VEH_0072""","""EMP_002""","""RUTA_018""","""EMP_002_DRV_002""",2024-03-04 08:23:24,2024-03-04 07:34:24,7.640372,-75.810033,33.01,59.61,91.36,43974.06,50.0,"""P20"""


In [43]:
invalid_coordinates = data_raw.filter(
    (pl.col("latitude") < -90) | (pl.col("latitude") > 90) |
    (pl.col("longitude") < -180) | (pl.col("longitude") > 180)
)
print(f"Dimensions: {invalid_coordinates.shape}")
invalid_coordinates.head()

Dimensions: (2484, 15)


tracking_id,vehicle_id,company_id,route_id,driver_id,event_time,received_at,latitude,longitude,speed,fuel_level,battery_level,odometer_km,passenger_count,event_type
str,str,str,str,str,datetime[μs],datetime[μs],f64,f64,f64,f64,f64,f64,f64,str
"""53ec2276-2279-40c9-8784-f7cb97…","""VEH_0180""","""EMP_005""","""RUTA_040""","""EMP_005_DRV_070""",2024-03-06 08:14:38,2024-03-06 08:14:43,200.0,-75.225749,36.15,91.0,81.74,8969.84,null,"""TR10"""
"""f4589aa3-5046-419e-b288-802442…","""VEH_0179""","""EMP_004""","""RUTA_016""","""EMP_004_DRV_067""",2024-03-06 09:38:17,2024-03-06 09:38:23,6.354395,300.0,44.62,90.82,70.09,67194.37,null,"""TR10"""
"""c89cea9d-2d30-4602-9a09-830310…","""VEH_0059""","""EMP_004""","""RUTA_031""","""EMP_004_DRV_032""",2024-03-05 10:57:23,2024-03-05 10:57:28,95.0,-73.35256,57.55,74.38,83.67,45877.09,null,"""TR10"""
"""3afb2886-d642-46b3-89b6-3bb97f…","""VEH_0130""","""EMP_005""","""RUTA_036""","""EMP_005_DRV_076""",2024-03-05 13:29:25,2024-03-05 13:29:26,-95.0,-73.86108,25.0,83.6,94.54,65191.81,null,"""TR10"""
"""0e3dcf2d-0bff-4091-8ea0-2facd6…","""VEH_0027""","""EMP_002""","""RUTA_030""","""EMP_002_DRV_025""",2024-03-04 15:53:35,2024-03-04 15:53:40,200.0,-74.545406,34.64,44.07,89.61,7487.87,null,"""T30"""


In [44]:
imposible_speed = data_raw.filter(
    (pl.col("speed") < 0) | (pl.col("speed") > 150)
)
print(f"Dimensions: {imposible_speed.shape}")
imposible_speed.head()

Dimensions: (2000, 15)


tracking_id,vehicle_id,company_id,route_id,driver_id,event_time,received_at,latitude,longitude,speed,fuel_level,battery_level,odometer_km,passenger_count,event_type
str,str,str,str,str,datetime[μs],datetime[μs],f64,f64,f64,f64,f64,f64,f64,str
"""2d505eb6-523f-43d7-b36b-b8db86…","""VEH_0175""","""EMP_005""","""RUTA_040""","""EMP_005_DRV_058""",2024-03-04 23:32:11,2024-03-04 23:32:14,7.049148,-73.228448,320.0,33.17,89.79,23073.61,null,"""T30"""
"""d6806ec3-331e-405b-a0a8-1042f5…","""VEH_0217""","""EMP_002""","""RUTA_006""","""EMP_002_DRV_017""",2024-03-04 17:07:17,2024-03-04 16:31:17,4.36243,-76.000481,200.0,69.48,86.72,74085.53,null,"""T30"""
"""08da269e-6be6-4aa1-a125-c73b68…","""VEH_0178""","""EMP_003""","""RUTA_002""","""EMP_003_DRV_049""",2024-03-05 01:23:48,2024-03-05 01:23:49,5.118087,-75.292002,160.0,46.88,96.06,29412.8,null,"""TR10"""
"""c816d704-a6a0-4e34-94d8-0a9f25…","""VEH_0025""","""EMP_005""","""RUTA_018""","""EMP_005_DRV_074""",2024-03-05 22:27:05,2024-03-05 22:27:06,4.896591,-75.745477,200.0,75.27,90.0,28089.23,null,"""T30"""
"""eeb313e2-ab76-4854-8a49-c6a6fd…","""VEH_0150""","""EMP_005""","""RUTA_018""","""EMP_005_DRV_055""",2024-03-05 17:23:55,2024-03-05 17:24:03,6.789343,-75.755147,160.0,78.41,76.55,44855.9,null,"""TR10"""


In [51]:
MINUTOS_UMBRAL = 5

late_events = (
    data_raw
    .with_columns(
        (pl.col("received_at") - pl.col("event_time")).alias("delay")
    )
    .filter(
        pl.col("delay") > pl.duration(minutes=MINUTOS_UMBRAL)
    )
    .sort("delay", descending=True)
)
print(f"Dimensions: {late_events.shape}")
late_events.head(10)

Dimensions: (7500, 16)


tracking_id,vehicle_id,company_id,route_id,driver_id,event_time,received_at,latitude,longitude,speed,fuel_level,battery_level,odometer_km,passenger_count,event_type,delay
str,str,str,str,str,datetime[μs],datetime[μs],f64,f64,f64,f64,f64,f64,f64,str,duration[μs]
"""d6313d28-3d6d-404b-9b7f-ccef35…","""VEH_0156""","""EMP_001""","""RUTA_007""","""EMP_001_DRV_036""",2024-03-05 14:44:15,2024-03-05 15:28:15,2.873921,-73.208739,42.36,64.39,97.25,19953.54,null,"""TR10""",44m
"""4bdc4504-1a7d-4692-83ba-59a792…",null,"""EMP_005""","""RUTA_023""","""EMP_005_DRV_050""",2024-03-04 18:46:16,2024-03-04 19:30:16,4.978445,-75.176211,32.83,30.78,75.38,55250.07,null,"""TR10""",44m
"""ba8399e1-e9c8-458c-a867-dfdfdd…","""VEH_0218""","""EMP_003""","""RUTA_028""","""EMP_003_DRV_045""",2024-03-06 09:05:56,2024-03-06 09:49:56,3.347129,-73.095361,57.93,72.25,71.13,46987.08,null,"""TR10""",44m
"""35b3a843-dd0e-48ea-b1bd-db58c0…","""VEH_0179""","""EMP_004""","""RUTA_016""","""EMP_004_DRV_073""",2024-03-05 13:04:45,2024-03-05 13:48:45,6.365622,-75.91197,28.9,92.3,82.84,66651.14,55.0,"""P20""",44m
"""506c3c93-902e-4e78-a7b1-bfb95a…","""VEH_0013""","""EMP_003""","""RUTA_014""","""EMP_003_DRV_044""",2024-03-05 01:16:10,2024-03-05 02:00:10,null,-73.779188,72.69,46.81,92.1,35897.55,null,"""T30""",44m
"""ace10a03-2ac9-4f87-b760-90b9e1…","""VEH_0006""","""EMP_001""","""RUTA_014""","""EMP_001_DRV_039""",2024-03-04 15:52:28,2024-03-04 16:36:28,8.89129,-75.926293,25.84,66.81,73.57,73009.6,null,"""T30""",44m
"""e739c43d-c117-4b2c-b548-ce68c3…","""VEH_0165""","""EMP_005""","""RUTA_021""","""EMP_005_DRV_034""",2024-03-05 15:04:14,2024-03-05 15:48:14,5.768195,-74.331925,28.34,72.33,82.96,12606.75,16.0,"""P20""",44m
"""341e1ac2-e50e-4b00-9ed0-c8662a…","""VEH_0205""","""EMP_005""","""RUTA_004""","""EMP_005_DRV_050""",2024-03-05 23:20:03,2024-03-06 00:04:03,5.826326,-73.616753,36.43,58.56,89.96,74605.99,null,"""T30""",44m
"""a5895370-9fca-4081-b0d4-aeaefa…","""VEH_0026""","""EMP_001""","""RUTA_011""","""EMP_001_DRV_059""",2024-03-04 22:19:57,2024-03-04 23:03:57,200.0,-75.58104,45.78,31.1,84.28,40201.18,null,"""TR10""",44m


In [52]:
weak_coverage_zones = (
    late_events
    .with_columns([
        pl.col("latitude").round(3).alias("zone_lat"),
        pl.col("longitude").round(3).alias("zone_lon")
    ])
    .group_by(["zone_lat", "zone_lon"])
    .agg([
        pl.len().alias("total_delayed_events"),
        pl.col("delay").mean().alias("average_delay")
    ])
    .sort("total_delayed_events", descending=True)
)

weak_coverage_zones.head(10)

zone_lat,zone_lon,total_delayed_events,average_delay
f64,f64,u32,duration[μs]
9.852,-75.516,3,31m
5.853,-73.54,3,31m
7.927,-75.331,3,26m 20s
3.035,-74.114,3,33m 20s
9.774,-75.247,2,30m 30s
6.614,-74.437,2,23m 30s
3.534,-75.908,2,25m
5.069,-75.96,2,31m
6.33,-73.14,2,18m


In [46]:
driver_changes = (
    data_raw
    .with_columns(
        pl.col("event_time").dt.date().alias("date")
    )
    .group_by(["vehicle_id", "date"])
    .agg(
        pl.col("driver_id").n_unique().alias("unique_drivers"),
        pl.col("driver_id").unique().alias("drivers_list")
    )
    .filter(pl.col("unique_drivers") > 1)
    .sort(["date", "vehicle_id"])
)

driver_changes.head()

vehicle_id,date,unique_drivers,drivers_list
str,date,u32,list[str]
null,null,6,"[""EMP_002_DRV_059"", ""EMP_003_DRV_012"", … ""EMP_002_DRV_034""]"
"""VEH_0001""",null,3,"[""EMP_001_DRV_049"", ""EMP_001_DRV_059"", ""EMP_001_DRV_024""]"
"""VEH_0003""",null,3,"[""EMP_003_DRV_048"", ""EMP_003_DRV_010"", ""EMP_003_DRV_060""]"
"""VEH_0004""",null,3,"[""EMP_004_DRV_024"", ""EMP_004_DRV_015"", ""EMP_004_DRV_050""]"
"""VEH_0005""",null,2,"[""EMP_005_DRV_041"", ""EMP_005_DRV_072""]"


In [54]:
odometer_resets = (
    data_raw
    .sort(["vehicle_id", "event_time"])
    .with_columns(
        pl.col("odometer_km").shift(1).over("vehicle_id").alias("previous_odometer")
    )

    .filter(
        (pl.col("odometer_km") < pl.col("previous_odometer")) &
        pl.col("previous_odometer").is_not_null()
    )
    .with_columns(
        (pl.col("previous_odometer") - pl.col("odometer_km")).alias("odometer_drop")
    )
)

print(f"Dimensions: {odometer_resets.shape}")
odometer_resets.head()

Dimensions: (1401, 17)


tracking_id,vehicle_id,company_id,route_id,driver_id,event_time,received_at,latitude,longitude,speed,fuel_level,battery_level,odometer_km,passenger_count,event_type,previous_odometer,odometer_drop
str,str,str,str,str,datetime[μs],datetime[μs],f64,f64,f64,f64,f64,f64,f64,str,f64,f64
"""cc078dfd-1418-4057-b941-4c3679…",null,"""EMP_003""","""RUTA_007""","""EMP_003_DRV_033""",null,2024-03-05 13:25:35,6.852745,-74.162644,64.94,80.14,74.09,20209.71,null,"""T30""",60843.87,40634.16
"""efc6ec3a-2524-48b1-a4c3-cfe3e2…",null,"""EMP_003""","""RUTA_009""","""EMP_003_DRV_012""",null,2024-03-04 11:17:02,6.607127,-74.433125,50.72,39.23,87.75,12667.73,null,"""TR10""",49275.68,36607.95
"""180f2422-2683-4f8f-9e8d-363bbc…",null,"""EMP_002""","""RUTA_013""","""EMP_002_DRV_059""",null,2024-03-04 17:34:53,2.466918,-73.525174,64.04,44.45,84.06,11713.89,null,"""TR10""",12667.73,953.84
"""dafbe468-0973-44aa-af6c-5fb4aa…",null,"""EMP_003""","""RUTA_017""","""EMP_003_DRV_037""",2024-03-04 08:00:47,2024-03-04 08:00:54,6.985728,-73.151799,57.67,45.34,70.85,43993.95,11.0,"""P20""",77587.9,33593.95
"""41c9c31a-a0e6-4313-bc53-b52fb1…",null,"""EMP_001""","""RUTA_033""","""EMP_001_DRV_003""",2024-03-04 08:09:34,2024-03-04 08:10:51,8.048451,-74.108407,17.35,91.72,98.1,74717.08,5.0,"""P20""",75682.99,965.91
